# Stage 2 Notebook 22 - Exp2Q Hybrid prior-generator + query-refiner

**Why this exists.** Across NB17-NB21 we collected two complementary partial wins:
- **Exp2N (priors)**: matched_iou=0.42 (best ever) but cls flat at 0.13 -> decoded_f1=0.011.
- **Exp2P (queries)**: val_lane_f1=0.65, pos-neg=+0.038 (cls finally works!) but matched_iou collapsed to 0.13 -> decoded_f1=0.026.

Neither paradigm gets close to CLRKDNet alone. **The big move: combine them.** Two-stage cascade pattern from Sparse R-CNN (Sun 2021) and Mask2Former (Cheng 2022):

- **Stage 1**: the proven CLRKDLaneHead produces 192 prior-based curves with good geometry.
- **Stage 2**: K=12 learned queries pass through a 2-layer transformer decoder that cross-attends over (a) the 192 per-prior features and (b) the merged spatial feature map. Each query outputs cls + refined curve.
- **Hungarian matches the K=12 outputs to GT.** Stage 1's cls is unused at inference; geometry is supervised end-to-end through stage 1's prior outputs.

Hypothesis: the K=12 query design owns the cls task (proven in Exp2P). The prior-based stage owns the geometry (proven in Exp2N). The decoder learns to pick/refine the right priors for each query slot.

Backbone (RMT + GCA + AIFI), detection head, eval pipeline (decoded_f1 + oracle_f1) all unchanged. The only change is the lane head (now `HybridPriorQueryHead`).

Reference: external_repos/CLRKDNet-master (the prior generator), DETR (Carion 2020), Sparse R-CNN (Sun et al. 2021), Mask2Former (Cheng et al. 2022).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through Hybrid head. lane_shape should be
# (1, K, 72, 2) where K=12 (or whatever num_queries the smoke harness lets through).
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp17_rmt_gca_hybrid_prior_query_joint_smoke.log
OK exp17_rmt_gca_hybrid_prior_query_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=2.2536 det_loss=3.4268 grad_cos=-0.0975 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5009090900421143, 'gate/lane_mean': 0.4966968297958374, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp17_rmt_gca_hybrid_prior_query_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp17_rmt_gca_hybrid_prior_query_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/data

0

## What to watch in Exp2Q training

Reference recent results:
- Exp2N (priors): matched_iou=0.42, decoded_f1=0.011, val_lane_f1=0.05.
- Exp2P (queries): matched_iou=0.13, decoded_f1=0.026, val_lane_f1=0.65.

Pass criteria at epoch 10 -- **the goal is to inherit BOTH wins simultaneously**:

- **`val/matched_line_iou >= 0.30`**: stage 1 priors should retain reasonable geometry (looser than Exp2N's 0.42 since training is now joint with stage 2).
- **`val/lane_exist_best_f1 >= 0.40`**: stage 2 queries should learn to rank, similar to Exp2P (0.67 was upper).
- **`val/lane/decoded_f1 >= 0.10`**, ideally >= 0.20: the metric that matters. Decoded top-K from stage 2 queries should land on geometry-good curves.
- **`val/lane/decoded_oracle_f1 >= 0.20`**: oracle ranking with stage 2's coord_pred should reflect the underlying geometry quality.

Failure signals -> next ablation:

- `decoded_f1 < 0.05` AND geometry held: queries failed to pick from priors. Add ROI-gather conditioning so queries see prior curves directly, not just per-prior pooled features.
- Geometry collapsed (matched_iou < 0.20): stage 1 supervision insufficient. Add explicit auxiliary geometry loss on stage 1 outputs (the `stage1_*` keys in the head output).
- Both fail: hybrid pattern itself doesn't help, query-only design is the path forward; prioritize Exp2R.